# Fitbit / Bellabeat Case Study: Data Cleaning

**Notebook:** `notebooks/data_cleaning.ipynb`  |  **Stage 1 of 4** (Cleaning → EDA → SQL analysis → Streamlit + report)

## Goal
Turn the raw Fitabase CSV files into a small set of **clean, analysis-ready tables** that the EDA notebook, the SQL queries and the Streamlit app can all use.

## Data
- **Source:** Fitabase Fitbit export, 12 Apr 2016 to 12 May 2016 (31 days)
- **Users:** 33 in the daily activity data (only 24 have sleep data and only 8 have weight/BMI data)
- **Scope decision:** the project is about **daily activity, sleep pattern and BMI**. The minute-level files and the heart-rate file are deliberately **not loaded** (too granular, and heart rate covers only 14 users).

## Tables used

| Raw file | Grain (one row = ...) | Used for |
|---|---|---|
| `dailyActivity_merged.csv` | one user, one day | steps, distance, calories, activity minutes |
| `sleepDay_merged.csv` | one user, one day | sleep duration, time in bed |
| `weightLogInfo_merged.csv` | one user, one weigh-in | weight and BMI |
| `hourlySteps / hourlyCalories / hourlyIntensities` | one user, one hour | time-of-day patterns |

## Steps in this notebook
1. Imports, paths and load the raw files
2. Inventory: rows, users, nulls, duplicates
3. Fix the date columns
4. Remove duplicates
5. Validity checks (suspicious or impossible values)
6. Daily activity: add feature columns
7. Sleep: add feature columns
8. BMI / weight: clean and categorise
9. Merge the three hourly tables
10. Export clean CSVs and the SQLite database

## Key decisions (worth remembering for interviews)
- **Tables are kept separate** because they have different grains and different numbers of users. They are joined in SQL only when a question needs it.
- **Not-worn days are flagged, not deleted** (`is_worn` column), so results can be shown with and without them.
- **BMI is analysed per user**, because 2 users hold 54 of the 67 weight records.

## Step 1: Imports, paths and loading the raw data

**What this does:** imports the libraries, defines where files live, and loads the 6 raw CSVs into one dictionary called `raw`.

**How it works**
- `pandas` is used for tables (DataFrames), `numpy` for number helpers, `sqlite3` for the database and `pathlib.Path` for file paths.
- The paths start with `..` because this notebook lives inside `notebooks/`, so `..` means "go up one folder" to reach `Data/`.
- `files` maps a short nickname (e.g. `daily_activity`) to the CSV file name.
- The dictionary comprehension reads every file, so `raw["sleep_daily"]` is the sleep table.
- `mkdir(exist_ok=True)` creates `Data/processed` and `db/` if they are missing.

**Expected output:** 6 lines with table shapes, e.g. `daily_activity (940, 15)`, `sleep_daily (413, 5)`, `weight_log (67, 8)`, and three hourly tables of `(22099, ...)`.

**If it fails:** a `FileNotFoundError` means a CSV is missing from `Data/raw/`, or the folder name is spelled differently (capital `D` in `Data`).

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
DB_PATH = Path("../db/fitbit.db")
PROCESSED.mkdir(exist_ok=True)
DB_PATH.parent.mkdir(exist_ok=True)

files = {
    "daily_activity":     "dailyActivity_merged.csv",
    "sleep_daily":        "sleepDay_merged.csv",
    "weight_log":         "weightLogInfo_merged.csv",
    "hourly_calories":    "hourlyCalories_merged.csv",
    "hourly_intensities": "hourlyIntensities_merged.csv",
    "hourly_steps":       "hourlySteps_merged.csv",
}

raw = {name: pd.read_csv(RAW / fname) for name, fname in files.items()}
for name, df in raw.items():
    print(f"{name:20s} {df.shape}")

daily_activity       (940, 15)
sleep_daily          (413, 5)
weight_log           (67, 8)
hourly_calories      (22099, 3)
hourly_intensities   (22099, 4)
hourly_steps         (22099, 3)


## Step 2: Data inventory (rows, users, nulls, duplicates)

**What this does:** builds one summary table describing every raw table before any cleaning.

**Why:** cleaning decisions should come from evidence. This table tells us what needs fixing.

**How it works:** for each table it records
- `rows`: number of records
- `unique_users`: how many different `Id` values (people) appear
- `null_cells`: total empty cells in the table
- `exact_duplicate_rows`: rows that are identical copies of another row

**What we found**
- Daily and hourly tables: **33 users**, no nulls, no duplicates.
- `sleep_daily`: only **24 users** and **3 duplicate rows**.
- `weight_log`: only **8 users** and **65 nulls** (all in the `Fat` column, confirmed in Step 5).

**Takeaway for the report:** sleep covers 73% of users and weight only 24%, so sleep and BMI conclusions apply to a subset of users.

In [2]:
summary = []
for name, df in raw.items():
    summary.append({
        "table": name,
        "rows": len(df),
        "unique_users": df["Id"].nunique(),
        "null_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
    })
pd.DataFrame(summary)

,table,rows,unique_users,null_cells,exact_duplicate_rows
0,daily_activity,940,33,0,0
1,sleep_daily,413,24,0,3
2,weight_log,67,8,65,0
3,hourly_calories,22099,33,0,0
4,hourly_intensities,22099,33,0,0
5,hourly_steps,22099,33,0,0


## Step 3: Fix the date columns

**What this does:** converts date columns from plain text (like `"4/12/2016 12:00:00 AM"`) into real datetime values, and gives the tables a common `date` column.

**Why:** text dates cannot be sorted, grouped by weekday or joined reliably. A shared `date` column lets us join daily activity, sleep and weight later on `Id` + `date`.

**How it works**
- `pd.to_datetime(..., format=...)` tells pandas exactly how to read the text. `%m/%d/%Y` is month/day/year and `%I:%M:%S %p` is 12-hour time with AM/PM.
- `.dt.normalize()` sets the time part to midnight, so a sleep record at "12:00 AM" becomes a pure date.
- Weight and hourly tables keep a full `datetime` (hourly needs the hour), and the weight table also gets a date-only `date`.

**Expected output:** all three tables show the range **2016-04-12 to 2016-05-12**.

**If it fails:** a `ValueError` about the format means the text does not match the pattern; print `df["column"].head()` to see the real format.

In [3]:
dt_fmt = "%m/%d/%Y %I:%M:%S %p"

raw["daily_activity"]["date"] = pd.to_datetime(raw["daily_activity"]["ActivityDate"], format="%m/%d/%Y")
raw["sleep_daily"]["date"] = pd.to_datetime(raw["sleep_daily"]["SleepDay"], format=dt_fmt).dt.normalize()

raw["weight_log"]["datetime"] = pd.to_datetime(raw["weight_log"]["Date"], format=dt_fmt)
raw["weight_log"]["date"] = raw["weight_log"]["datetime"].dt.normalize()

for name in ["hourly_calories", "hourly_intensities", "hourly_steps"]:
    raw[name]["datetime"] = pd.to_datetime(raw[name]["ActivityHour"], format=dt_fmt)

for name in ["daily_activity", "sleep_daily", "weight_log"]:
    print(name, raw[name]["date"].min().date(), "->", raw[name]["date"].max().date())

daily_activity 2016-04-12 -> 2016-05-12
sleep_daily 2016-04-12 -> 2016-05-12
weight_log 2016-04-12 -> 2016-05-12


## Step 4: Remove duplicates

**What this does:** drops the exact duplicate rows in `sleep_daily`, then checks that every table has at most one row per user per day.

**How it works**
- `drop_duplicates()` keeps the first copy of identical rows. `reset_index(drop=True)` renumbers the rows 0, 1, 2, ...
- `duplicated(subset=["Id", "date"])` counts rows that share the same user and day. This must be 0, otherwise later joins would create extra rows.

**Expected output:** `sleep_daily: 413 -> 410 rows`, and **0** duplicates for all three tables.

**Report line:** "Removed 3 duplicate rows from the sleep table."

In [4]:
before = len(raw["sleep_daily"])
raw["sleep_daily"] = raw["sleep_daily"].drop_duplicates().reset_index(drop=True)
print(f"sleep_daily: {before} -> {len(raw['sleep_daily'])} rows")

print("(Id, date) duplicates:")
print("  daily_activity:", raw["daily_activity"].duplicated(subset=["Id", "date"]).sum())
print("  sleep_daily:   ", raw["sleep_daily"].duplicated(subset=["Id", "date"]).sum())
print("  weight_log:    ", raw["weight_log"].duplicated(subset=["Id", "date"]).sum())

sleep_daily: 413 -> 410 rows
(Id, date) duplicates:
  daily_activity: 0
  sleep_daily:    0
  weight_log:     0


## Step 5: Validity checks (looking for suspicious or impossible values)

**What this does:** only **checks**. Nothing is changed in this cell. The results decide what we do in Steps 6 to 8.

**Checks and findings**

| Check | Result | Decision |
|---|---|---|
| Days with 0 steps / 1,440 sedentary minutes | 77 / 79 days | look closer |
| **0 steps AND 1,440 sedentary minutes** | **72 days, 15 users** | Tracker was almost certainly not worn. **Flag it** in Step 6 |
| Sleep rows where minutes asleep > time in bed | 0 | Nothing to fix |
| Weight and BMI ranges | 52.6 to 133.5 kg, BMI 21.5 to 47.5 | Plausible, keep everything |
| Nulls in `weight_log` | 65, **all in `Fat`** | Drop the `Fat` column in Step 8 |

**Why 1,440?** A day has 24 h x 60 min = 1,440 minutes. If every minute is "sedentary" and there are zero steps, the device was off the wrist rather than the person sitting still all day.

**BMI outlier note:** the maximum BMI of 47.54 belongs to one user with a single entry. It is a real, plausible value, so it stays.

In [5]:
da = raw["daily_activity"]
print("Days with 0 steps:", (da["TotalSteps"] == 0).sum())
print("Days with 1440 sedentary min:", (da["SedentaryMinutes"] == 1440).sum())
not_worn = (da["TotalSteps"] == 0) & (da["SedentaryMinutes"] == 1440)
print("Likely not-worn days (0 steps AND 1440 sedentary):", not_worn.sum())
print("Users with a not-worn day:", da.loc[not_worn, "Id"].nunique())

sl = raw["sleep_daily"]
print("Sleep rows where asleep > time in bed:", (sl["TotalMinutesAsleep"] > sl["TotalTimeInBed"]).sum())

w = raw["weight_log"]
print(w[["WeightKg", "BMI"]].describe().round(2))
print("Nulls per column in weight_log:"); print(w.isna().sum())

Days with 0 steps: 77
Days with 1440 sedentary min: 79
Likely not-worn days (0 steps AND 1440 sedentary): 72
Users with a not-worn day: 15
Sleep rows where asleep > time in bed: 0
       WeightKg    BMI
count     67.00  67.00
mean      72.04  25.19
std       13.92   3.07
min       52.60  21.45
25%       61.40  23.96
50%       62.50  24.39
75%       85.05  25.56
max      133.50  47.54
Nulls per column in weight_log:
Id                 0
Date               0
WeightKg           0
WeightPounds       0
Fat               65
BMI                0
IsManualReport     0
LogId              0
datetime           0
date               0
dtype: int64


## Step 6: Daily activity, add feature columns

**What this does:** creates the cleaned `daily` table with extra columns the analysis needs.

**New columns**

| Column | Meaning |
|---|---|
| `is_worn` | `False` for the 72 not-worn days, `True` otherwise. **Flagged, not deleted** |
| `day_of_week` | Monday, Tuesday, ... (for weekday patterns) |
| `is_weekend` | `True` for Saturday and Sunday |
| `total_active_minutes` | very active + fairly active + lightly active minutes |
| `activity_level` | step-based bucket: Sedentary (<5,000), Low active (5,000-7,499), Somewhat active (7,500-9,999), Active (10,000-12,499), Highly active (12,500+) |

**How `pd.cut` works:** it sorts a number into a labelled range. The bins `[-1, 4999, 7499, 9999, 12499, inf]` create 5 ranges, and `labels` names them. The `-1` makes sure 0 steps is included.

**Expected output:** 868 worn vs 72 not worn; activity levels 303 / 171 / 163 / 159 / 144.

**Using this later:** for averages of steps or calories, filter with `is_worn = True` (pandas) or `WHERE is_worn = 1` (SQL).

In [6]:
daily = raw["daily_activity"].copy()

daily["is_worn"] = ~((daily["TotalSteps"] == 0) & (daily["SedentaryMinutes"] == 1440))
daily["day_of_week"] = daily["date"].dt.day_name()
daily["is_weekend"] = daily["date"].dt.dayofweek >= 5
daily["total_active_minutes"] = (daily["VeryActiveMinutes"] + daily["FairlyActiveMinutes"] + daily["LightlyActiveMinutes"])
daily["activity_level"] = pd.cut(
    daily["TotalSteps"],
    bins=[-1, 4999, 7499, 9999, 12499, np.inf],
    labels=["Sedentary", "Low active", "Somewhat active", "Active", "Highly active"],
)
daily = daily.drop(columns=["ActivityDate"])

print(daily["is_worn"].value_counts())
print(daily["activity_level"].value_counts().sort_index())

is_worn
True     868
False     72
Name: count, dtype: int64
activity_level
Sedentary          303
Low active         171
Somewhat active    163
Active             159
Highly active      144
Name: count, dtype: int64


## Step 7: Sleep pattern, add feature columns

**What this does:** creates the cleaned `sleep` table with easier-to-read columns.

**New columns**

| Column | Meaning |
|---|---|
| `hours_asleep`, `hours_in_bed` | minutes divided by 60 |
| `sleep_efficiency` | minutes asleep / minutes in bed. 0.94 means asleep for 94% of the time in bed |
| `day_of_week`, `is_weekend` | for weekday vs weekend sleep comparisons |
| `sleep_category` | Under 6h, 6-9h (recommended), Over 9h |

**Detail:** `right=False` in `pd.cut` makes the ranges "6 up to but not including 9", so exactly 9.0 hours falls in "Over 9h".

**Expected output:** 271 nights in the recommended range, 100 under 6h, 39 over 9h. Median efficiency about **0.943**, minimum 0.498.

**Limitation:** only 24 users have sleep data.

In [7]:
sleep = raw["sleep_daily"].copy()

sleep["hours_asleep"] = (sleep["TotalMinutesAsleep"] / 60).round(2)
sleep["hours_in_bed"] = (sleep["TotalTimeInBed"] / 60).round(2)
sleep["sleep_efficiency"] = (sleep["TotalMinutesAsleep"] / sleep["TotalTimeInBed"]).round(3)
sleep["day_of_week"] = sleep["date"].dt.day_name()
sleep["is_weekend"] = sleep["date"].dt.dayofweek >= 5
sleep["sleep_category"] = pd.cut(
    sleep["hours_asleep"], bins=[0, 6, 9, 24],
    labels=["Under 6h", "6-9h (recommended)", "Over 9h"], right=False,
)
sleep = sleep.drop(columns=["SleepDay"])

print(sleep["sleep_category"].value_counts())
print(sleep["sleep_efficiency"].describe().round(3))

sleep_category
6-9h (recommended)    271
Under 6h              100
Over 9h                39
Name: count, dtype: int64
count    410.000
mean       0.916
std        0.087
min        0.498
25%        0.912
50%        0.943
75%        0.961
max        1.000
Name: sleep_efficiency, dtype: float64


## Step 8: BMI / weight table

**What this does:** cleans the weight log and adds a BMI category.

**Steps**
- Drop `Fat` (65 of 67 values empty), `Date` (replaced by our cleaner `date`) and `LogId` (not needed).
- `IsManualReport` becomes 1/0 instead of True/False (easier in SQL).
- `bmi_category` uses the standard WHO cut-offs: under 18.5 Underweight, 18.5 to <25 Normal, 25 to <30 Overweight, 30+ Obese.

**Expected output:** 34 Normal, 32 Overweight, 1 Obese, 0 Underweight (these are **records**, not people).

**Important limitation:** the last output shows the records per user. Two users have 30 and 24 records (54 of 67 in total) while three users have only 1 or 2. Therefore **BMI must be analysed per user** (average BMI per `Id`), otherwise two people dominate the result.

In [8]:
weight = raw["weight_log"].copy()
weight = weight.drop(columns=["Fat", "Date", "LogId"])
weight["IsManualReport"] = weight["IsManualReport"].astype(int)
weight["bmi_category"] = pd.cut(
    weight["BMI"], bins=[0, 18.5, 25, 30, np.inf],
    labels=["Underweight", "Normal", "Overweight", "Obese"], right=False,
)

print(weight["bmi_category"].value_counts())
print(weight.groupby("Id").size().sort_values(ascending=False))

bmi_category
Normal         34
Overweight     32
Obese           1
Underweight     0
Name: count, dtype: int64
Id
6962181067    30
8877689391    24
4558609924     5
1503960366     2
2873212765     2
4319703577     2
1927972279     1
5577150313     1
dtype: int64


## Step 9: Merge the three hourly tables

**What this does:** combines hourly steps, calories and intensity into one `hourly` table.

**Why merging is safe here:** the three tables have the **same grain** (one user, one hour), so they can be lined up side by side. This is different from daily vs sleep vs weight, which have different grains and different numbers of users, so those stay separate.

**How it works**
- `.merge(..., on=["Id", "datetime"], how="inner")` matches rows with the same user and hour and keeps only rows present in both tables.
- We then extract `date`, `hour` (0 to 23), `day_of_week` and `is_weekend` for time-of-day charts.

**Expected output:** `(22099, 10)`. The row count equals the original, which proves the merge lost nothing.

In [9]:
hourly = (
    raw["hourly_steps"][["Id", "datetime", "StepTotal"]]
    .merge(raw["hourly_calories"][["Id", "datetime", "Calories"]], on=["Id", "datetime"], how="inner")
    .merge(raw["hourly_intensities"][["Id", "datetime", "TotalIntensity", "AverageIntensity"]], on=["Id", "datetime"], how="inner")
)
hourly["date"] = hourly["datetime"].dt.normalize()
hourly["hour"] = hourly["datetime"].dt.hour
hourly["day_of_week"] = hourly["datetime"].dt.day_name()
hourly["is_weekend"] = hourly["datetime"].dt.dayofweek >= 5

print(hourly.shape, "(expected 22099 rows)")

(22099, 10) (expected 22099 rows)


## Step 10: Export the clean data (CSV files and SQLite database)

**What this does:** saves the 4 clean tables in two formats.

| Output | Location | Used by |
|---|---|---|
| CSV files | `Data/processed/` | EDA notebook (pandas) |
| SQLite database | `db/fitbit.db` | SQL queries and the Streamlit app |

**Tables saved:** `daily_activity` (940 x 20), `sleep_daily` (410 x 11), `weight_log` (67 x 8), `hourly_activity` (22,099 x 10).

**Details**
- `if_exists="replace"` lets you re-run the notebook safely; the tables are rebuilt each time.
- The small loop converts pandas `category` columns to plain text, because SQLite cannot store that type.
- The last two queries prove the database works: 4 tables exist, and `daily_activity` has 940 rows and 33 users.

**Reminders for SQL:** dates are stored as text like `2016-04-12 00:00:00` (use `date()` or `strftime()`), and `is_worn` is stored as 1/0.

In [10]:
clean_tables = {
    "daily_activity": daily,
    "sleep_daily": sleep,
    "weight_log": weight,
    "hourly_activity": hourly,
}

for name, df in clean_tables.items():
    df.to_csv(PROCESSED / f"{name}.csv", index=False)
    print(f"saved {name}.csv  {df.shape}")

with sqlite3.connect(DB_PATH) as conn:
    for name, df in clean_tables.items():
        out = df.copy()
        for col in out.columns:
            if str(out[col].dtype) == "category":
                out[col] = out[col].astype(str)
        out.to_sql(name, conn, if_exists="replace", index=False)
    print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
    print(pd.read_sql("SELECT COUNT(*) AS n, COUNT(DISTINCT Id) AS users FROM daily_activity", conn))

saved daily_activity.csv  (940, 20)
saved sleep_daily.csv  (410, 11)
saved weight_log.csv  (67, 8)
saved hourly_activity.csv  (22099, 10)
              name
0   daily_activity
1      sleep_daily
2       weight_log
3  hourly_activity
     n  users
0  940     33
